# Notification Priority Classification — Evaluation

**Week 9 | Authors:** Oliver Holmes, Sofia Bonoan, Denis Sokolov, Gonzalo Fernández  
**Project:** Notification Priority Classification Using Semantic Analysis

This notebook evaluates and compares all trained models on the held-out test set.
It is designed to run **incrementally** — you can run it now with just the baseline,
and re-run it once BERT/RoBERTa results arrive from the cluster.

Evaluation covers two aspects from the paper (§III.I):

1. **Classification metrics** — accuracy, macro-F1, per-class precision/recall/F1,
   and confusion matrices for every available model
2. **Ranking evaluation** — `PriorityScore = P(High) + 0.5·P(Medium)` sorted
   descending, evaluated with `Precision@k` (k = 5, 10, 20)

---

## 1. Imports & Setup

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
FIGURES_DIR  = PROJECT_ROOT / 'results' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

LABEL_ORDER    = ['high', 'medium', 'low']
LABEL_DISPLAY  = {'high': 'High', 'medium': 'Medium', 'low': 'Low'}
URGENCY_COLORS = {'high': '#E74C3C', 'medium': '#F39C12', 'low': '#2ECC71'}
MODEL_COLORS   = ['#2C3E50', '#2980B9', '#8E44AD', '#16A085']  # baseline, bert, roberta, ...

plt.rcParams.update({'figure.dpi': 150, 'font.size': 11, 'axes.titlesize': 12})
print('Setup complete.')

## 2. Load Available Model Predictions

The notebook auto-detects which `test_predictions.csv` files exist.
BERT and RoBERTa appear here once you copy the cluster results:
```bash
scp -r nlp10@10.205.20.10:~/nlp-project/results/ ./results/
```

In [ ]:
# Register models: display name -> path to test_predictions.csv
MODEL_PATHS = {
    'Baseline (TF-IDF + LR)': PROJECT_ROOT / 'results' / 'baseline'            / 'test_predictions.csv',
    'BERT':                    PROJECT_ROOT / 'results' / 'bert-base-uncased'   / 'test_predictions.csv',
    'RoBERTa':                 PROJECT_ROOT / 'results' / 'roberta-base'        / 'test_predictions.csv',
}

models = {}   # name -> DataFrame
print('Model predictions found:')
for name, path in MODEL_PATHS.items():
    if path.exists():
        df = pd.read_csv(path)
        df['true_label'] = df['true_label'].str.lower().str.strip()
        df['pred_label'] = df['pred_label'].str.lower().str.strip()
        models[name] = df
        print(f'  [OK]  {name:<30}  ({len(df)} test samples)')
    else:
        print(f'  [--]  {name:<30}  not found yet -> {path}')

if not models:
    raise FileNotFoundError('No predictions found. Run notebooks/03_modeling.ipynb first.')

MODEL_NAMES = list(models.keys())
print(f'\nEvaluating {len(MODEL_NAMES)} model(s): {MODEL_NAMES}')

---
## 3. Classification Metrics

We use **macro-F1 as the primary metric** because it treats all three urgency classes
equally, which is important given the mild class imbalance in our dataset.

In [ ]:
def compute_metrics(df):
    """Return a dict of aggregate and per-class metrics for one model's predictions."""
    y_true, y_pred = df['true_label'], df['pred_label']
    return {
        'accuracy':    round(accuracy_score(y_true, y_pred), 4),
        'macro_f1':    round(f1_score(y_true, y_pred, average='macro',     zero_division=0), 4),
        'macro_prec':  round(precision_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'macro_rec':   round(recall_score(y_true, y_pred, average='macro',  zero_division=0), 4),
        'per_class_f1': {
            lbl: round(f1_score(y_true, y_pred, labels=[lbl], average='micro', zero_division=0), 4)
            for lbl in LABEL_ORDER
        },
    }


all_metrics = {name: compute_metrics(df) for name, df in models.items()}

print('\n===== CLASSIFICATION METRICS (test set) =====')
header = f'{"Model":<30}  {"Accuracy":>9}  {"Macro-F1":>9}  {"Macro-P":>8}  {"Macro-R":>8}'
print(header)
print('-' * len(header))
for name, m in all_metrics.items():
    print(f'{name:<30}  {m["accuracy"]:>9.4f}  {m["macro_f1"]:>9.4f}  '
          f'{m["macro_prec"]:>8.4f}  {m["macro_rec"]:>8.4f}')

In [ ]:
# Full per-class report for each model
for name, df in models.items():
    print(f'\n--- {name} ---')
    print(classification_report(
        df['true_label'], df['pred_label'],
        labels=LABEL_ORDER,
        target_names=['High', 'Medium', 'Low'],
        zero_division=0,
    ))

### 3.1 Confusion Matrices

In [ ]:
n_models = len(models)
fig, axes = plt.subplots(1, n_models, figsize=(5.5 * n_models, 5))
if n_models == 1:
    axes = [axes]

tick_labels = ['High', 'Medium', 'Low']

for ax, (name, df) in zip(axes, models.items()):
    cm = confusion_matrix(df['true_label'], df['pred_label'], labels=LABEL_ORDER)
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(3)); ax.set_xticklabels(tick_labels, fontsize=10)
    ax.set_yticks(range(3)); ax.set_yticklabels(tick_labels, fontsize=10)
    ax.set_xlabel('Predicted');  ax.set_ylabel('True')
    macro_f1 = all_metrics[name]['macro_f1']
    ax.set_title(f'{name}\nmacro-F1 = {macro_f1:.4f}', fontweight='bold')
    for i in range(3):
        for j in range(3):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=13, fontweight='bold', color=color)

fig.suptitle('Confusion Matrices — All Models (Test Set)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
for ext in ('svg', 'png', 'pdf'):
    fig.savefig(FIGURES_DIR / f'fig11_confusion_matrices.{ext}',
                bbox_inches='tight', format=ext)
plt.show()
print('Figure 11 saved.')

### 3.2 Per-Class F1 Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

n_classes = 3
x = np.arange(n_classes)
bar_width = 0.7 / n_models
offsets = np.linspace(-(n_models - 1) / 2, (n_models - 1) / 2, n_models) * bar_width

for i, (name, m) in enumerate(all_metrics.items()):
    f1_vals = [m['per_class_f1'][lbl] for lbl in LABEL_ORDER]
    bars = ax.bar(x + offsets[i], f1_vals, bar_width * 0.9,
                  label=name, color=MODEL_COLORS[i], alpha=0.85)
    for bar, val in zip(bars, f1_vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(['High', 'Medium', 'Low'], fontsize=12)
ax.set_ylabel('F1-Score')
ax.set_ylim(0, 1.15)
ax.set_title('Per-Class F1-Score by Model (Test Set)', fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
for ext in ('svg', 'png', 'pdf'):
    fig.savefig(FIGURES_DIR / f'fig12_per_class_f1.{ext}',
                bbox_inches='tight', format=ext)
plt.show()
print('Figure 12 saved.')

---
## 4. Ranking Evaluation

A classifier that assigns accurate probability scores can also **rank** notifications
so the most urgent ones appear first.  We compute:

$$\text{PriorityScore} = P(\text{High}) + 0.5 \cdot P(\text{Medium})$$

and evaluate with **Precision@k** — the fraction of truly High-urgency notifications
in the top-k ranked positions.  A perfect ranker scores 1.0 for all k values.

In [ ]:
def priority_score(df):
    """PriorityScore = P(High) + 0.5 * P(Medium), as defined in the paper."""
    return df['prob_high'] + 0.5 * df['prob_medium']


def precision_at_k(df, k):
    """Fraction of truly High notifications in the top-k by PriorityScore."""
    ranked = df.assign(priority=priority_score(df)).sort_values('priority', ascending=False)
    top_k  = ranked.head(k)
    return (top_k['true_label'] == 'high').mean()


K_VALUES = [5, 10, 20]

print('===== RANKING EVALUATION — Precision@k =====')
header = f'{"Model":<30}' + ''.join(f'  P@{k:>2}' for k in K_VALUES)
print(header)
print('-' * len(header))
ranking_results = {}
for name, df in models.items():
    scores = {k: precision_at_k(df, k) for k in K_VALUES}
    ranking_results[name] = scores
    row = f'{name:<30}' + ''.join(f'  {scores[k]:.3f}' for k in K_VALUES)
    print(row)

# Baseline reference: random ranking
n_high = sum(1 for d in models[MODEL_NAMES[0]]['true_label'] if d == 'high')
n_total = len(models[MODEL_NAMES[0]])
random_p = n_high / n_total
print(f'\n  Random baseline (P(High) in test set): {random_p:.3f}')

### 4.1 Precision@k Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

k_range = list(range(1, len(models[MODEL_NAMES[0]]) + 1))
for i, (name, df) in enumerate(models.items()):
    ranked = df.assign(priority=priority_score(df)).sort_values('priority', ascending=False)
    p_at_k = [
        (ranked.head(k)['true_label'] == 'high').mean()
        for k in k_range
    ]
    ax.plot(k_range, p_at_k, color=MODEL_COLORS[i], linewidth=2, label=name)

# Random baseline
ax.axhline(random_p, color='grey', linestyle='--', linewidth=1.2, label=f'Random ({random_p:.2f})')

# Mark the k values from the table
for k in K_VALUES:
    ax.axvline(k, color='lightgrey', linestyle=':', linewidth=1)
    ax.text(k + 0.3, 0.02, f'k={k}', fontsize=8, color='grey')

ax.set_xlabel('k (top-k notifications)')
ax.set_ylabel('Precision@k (fraction truly High)')
ax.set_title('Ranking Quality: Precision@k Curve', fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
for ext in ('svg', 'png', 'pdf'):
    fig.savefig(FIGURES_DIR / f'fig13_precision_at_k.{ext}',
                bbox_inches='tight', format=ext)
plt.show()
print('Figure 13 saved.')

### 4.2 Ranked Notification Strip

Visual check: notifications sorted by PriorityScore (best model), coloured by true label.
A good ranker keeps red (High) at the top and green (Low) at the bottom.

In [ ]:
# Use the model with the highest macro-F1 for the strip visualisation
best_model_name = max(all_metrics, key=lambda n: all_metrics[n]['macro_f1'])
best_df = models[best_model_name].copy()
best_df['priority'] = priority_score(best_df)
best_df = best_df.sort_values('priority', ascending=False).reset_index(drop=True)

# Show top-30 notifications as a coloured bar strip
TOP_STRIP = min(30, len(best_df))
strip_df  = best_df.head(TOP_STRIP)

fig, ax = plt.subplots(figsize=(14, 3))
for rank, row in strip_df.iterrows():
    color = URGENCY_COLORS[row['true_label']]
    ax.bar(rank, 1, color=color, edgecolor='white', linewidth=0.5)

# Legend patches
import matplotlib.patches as mpatches
patches = [mpatches.Patch(color=URGENCY_COLORS[l], label=l.capitalize()) for l in LABEL_ORDER]
ax.legend(handles=patches, fontsize=9, loc='upper right')
ax.set_xlim(-0.5, TOP_STRIP - 0.5)
ax.set_ylim(0, 1.2)
ax.set_xlabel(f'Rank (sorted by PriorityScore, best model: {best_model_name})')
ax.set_yticks([])
ax.set_title(f'Top-{TOP_STRIP} Notifications by PriorityScore — True Labels\n'
             f'(Red=High, Orange=Medium, Green=Low)', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

plt.tight_layout()
for ext in ('svg', 'png', 'pdf'):
    fig.savefig(FIGURES_DIR / f'fig14_ranking_strip.{ext}',
                bbox_inches='tight', format=ext)
plt.show()
print('Figure 14 saved.')

---
## 5. Full Results Table

Consolidated table for Section IV of the paper.

In [ ]:
rows = []
for name, m in all_metrics.items():
    row = {
        'Model':         name,
        'Accuracy':      m['accuracy'],
        'Macro-F1':      m['macro_f1'],
        'Macro-Prec':    m['macro_prec'],
        'Macro-Rec':     m['macro_rec'],
        'F1-High':       m['per_class_f1']['high'],
        'F1-Medium':     m['per_class_f1']['medium'],
        'F1-Low':        m['per_class_f1']['low'],
        'P@5':           ranking_results[name][5],
        'P@10':          ranking_results[name][10],
        'P@20':          ranking_results[name][20],
    }
    rows.append(row)

results_table = pd.DataFrame(rows).set_index('Model')

print('\n===== FULL RESULTS TABLE =====')
print(results_table.to_string())

# Save as CSV for easy copy-paste into Overleaf
table_path = PROJECT_ROOT / 'results' / 'results_table.csv'
results_table.to_csv(table_path)
print(f'\nTable saved -> {table_path}')

---
## 6. Evaluation Summary

In [ ]:
best_f1_model = max(all_metrics, key=lambda n: all_metrics[n]['macro_f1'])
best_f1_score = all_metrics[best_f1_model]['macro_f1']

print('=' * 55)
print('EVALUATION SUMMARY')
print('=' * 55)
print(f'  Models evaluated   : {len(models)}')
print(f'  Test samples       : {len(next(iter(models.values())))}')
print()
print(f'  Best macro-F1      : {best_f1_score:.4f}  ({best_f1_model})')
print()
print('  Figures saved:')
for n in ('fig11_confusion_matrices', 'fig12_per_class_f1',
          'fig13_precision_at_k', 'fig14_ranking_strip'):
    print(f'    results/figures/{n}.*')
print()
if len(models) < len(MODEL_PATHS):
    missing = [n for n in MODEL_PATHS if n not in models]
    print(f'  Pending results (re-run after scp from cluster):')
    for m in missing:
        print(f'    - {m}')
else:
    print('  All models evaluated. Results are complete.')
print('=' * 55)